# Prediction Model Tutorial

This notebook demonstrates the use case for a lightweight RNN model class. The class is intended to be used in prediction phase only, after training has been completed.
* Inputs: a config file and a set of trained model weights
* The class builds a model based on the architecture in the config file and loads the set of weights

In [ ]:
import sys
import joblib
import numpy as np
import matplotlib.pyplot as plt
sys.path.append("../src")
from models.moisture_rnn_operational import OperationalRNNPredictor
from models.moisture_rnn import scale_3d
from utils import read_yml, Dict

In [ ]:
params  = Dict(read_yml("../tests/test_files/params.yaml"))
weights_path   = "../tests/test_files/rnn.weights.h5"
scaler  = joblib.load("../tests/test_files/scaler.joblib")

## Build model from config and weights

In [ ]:
rnn = OperationalRNNPredictor.from_weights(params=params, weights_path = weights_path)

In [ ]:
rnn.summary()

## Predict

Test input with constant zeros (mean for standard scaled data). 
- Batch size 100 (10x10 grid)
- Sequence length 54 (48hr prediction + 6hr spinup)
- Number of features is required to match architecture

In [ ]:
nbatch = 100
nseq = 48
spinup = 6

X0 = np.zeros(shape=(nbatch, nseq+spinup, rnn.input_shape[2]))

In [ ]:
preds = rnn.predict(x=X0)

In [ ]:
fig, ax = plt.subplots()

ax.plot(preds[0,:,:])
ax.axvline(x=spinup, linestyle="dashed", color="k")
ax.annotate(text="Spinup Stop", xy=(spinup, 13.5), rotation=90, horizontalalignment="right")

plt.title("RNN Prediction with Constant Zeros Input")
plt.xlabel("Time Step")
plt.ylabel("FM10 (%)")

## Test Scalers

Use basic reshapes and a utility `scale_3d` to check transform and inverse transform of inputs

In [ ]:
# Reshape 3d -> 2d array, collapse batches and timesteps together
X0_2d = X0.reshape(-1, X0.shape[-1])

# Use scaler to transform back to original (unscaled) units
X_2d = scaler.inverse_transform(X0_2d)

# Reshape 2d -> 3d array to get back batches
X = X_2d.reshape(X0.shape)

In [ ]:
# Test 3d scale back to zeros, should be zero or machine epsilon
print(f"Max Difference: {np.max(np.abs(X0 - scale_3d(X=X, scaler=scaler)))}")

## Test Simple Mapping

Make a basic map assuming 10x10 grid with inputs and prediction. Mapping Ed and FM10 prediction from last time step in each case

In [ ]:
id_Ed = params.features_list.index("Ed")
img_Ed = X[:, -1, id_Ed].reshape(10, 10)

fig, ax = plt.subplots()
im = ax.imshow(img_Ed, vmin=0, vmax=20)

ax.set_xticks(np.arange(-0.5, 10, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 10, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1)

plt.colorbar(im)
plt.title("Constant Drying Equilibrium at end time")
plt.show()

In [ ]:
img_FM10 = preds.squeeze()[:,-1].reshape(10,10)

fig, ax = plt.subplots()
im = ax.imshow(img_FM10, vmin=0, vmax=20)

ax.set_xticks(np.arange(-0.5, 10, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 10, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1)

plt.colorbar(im)
plt.title("FM10 Prediction at end time")
plt.show()